<a href="https://colab.research.google.com/github/nbvhung/identifyApplications_AI/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#ô code 1
import pandas as pd
from google.colab import drive

# 1. Kết nối Drive (Bắt buộc chạy lại nếu vừa bị ngắt kết nối)
drive.mount('/content/drive')

# 2. Đọc file Parquet (Thay vì CSV)
# Đường dẫn này phải khớp với cái đường dẫn em vừa lưu ở ảnh 065765.png
path_parquet = '/content/drive/MyDrive/datasetAI/data_btl.parquet'

df = pd.read_parquet(path_parquet)

# 3. Lấy mẫu 100k dòng để train cho nhẹ máy (tùy Mạnh chọn)
df = df.sample(100000, random_state=42).reset_index(drop=True)

print(f"Số lượng ứng dụng: {df['ProtocolName'].nunique()}")


Mounted at /content/drive
Số lượng ứng dụng: 50


In [6]:
#ô code 2
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Mã hóa nhãn: Biến 'YOUTUBE', 'FACEBOOK'... thành số 0, 1, 2...
le = LabelEncoder()
df['ProtocolName_Encoded'] = le.fit_transform(df['ProtocolName'])

# 2. Chọn các cột đặc trưng (Features) - Loại bỏ các cột không phải số hoặc không cần thiết
# Mình tạm loại bỏ các cột IP và ID vì chúng dễ làm AI bị "học vẹt"
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ['ProtocolName_Encoded', 'L7Protocol']]

X = df[features]
y = df['ProtocolName_Encoded']

# 3. Xử lý giá trị vô hạn (Inf) hoặc lỗi dữ liệu thường gặp trong lưu lượng mạng
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# 4. Chia dữ liệu: 80% để học, 20% để kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Chuẩn hóa: Đưa các con số về cùng một thang đo (0-1 hoặc tương đương)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Đã chuẩn bị xong dữ liệu cho {len(features)} đặc trưng và {df['ProtocolName'].nunique()} ứng dụng.")


Đã chuẩn bị xong dữ liệu cho 80 đặc trưng và 50 ứng dụng.


In [7]:
#ô code 3
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

# 1. Tái mã hóa nhãn riêng cho tập Train và Test để đảm bảo tính liên tục (0, 1, 2...)
# Bước này cực kỳ quan trọng để sửa lỗi "Invalid classes inferred"
le_final = LabelEncoder()
y_train_fixed = le_final.fit_transform(y_train)

# Đối với tập Test, chúng ta chỉ lấy những nhãn mà tập Train đã học được
# Những nhãn nào tập Train không có sẽ bị loại bỏ ở tập Test để không gây lỗi
test_mask = y_test.isin(le_final.classes_)
X_test_fixed = X_test_scaled[test_mask]
y_test_fixed = le_final.transform(y_test[test_mask])

# 2. Khởi tạo mô hình XGBoost
model_baseline = XGBClassifier(
    n_estimators=100,
    random_state=42,
    tree_method='hist', # Sử dụng thuật toán histogram
    device='cuda',      # <--- CHỖ QUAN TRỌNG: Bật GPU cho XGBoost bản mới
    n_jobs=-1
)

# 3. Huấn luyện mô hình
print(f"Đang huấn luyện mô hình cơ sở trên {len(le_final.classes_)} ứng dụng...")
model_baseline.fit(X_train_scaled, y_train_fixed)

# 4. Dự đoán và đánh giá
y_pred = model_baseline.predict(X_test_fixed)
print(f"\n--- KẾT QUẢ HOÀN THÀNH WEEK 3 ---")
print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_fixed, y_pred):.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_test_fixed, y_pred))

Đang huấn luyện mô hình cơ sở trên 50 ứng dụng...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:16:19] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:16:19] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


KeyboardInterrupt: 

In [5]:
# SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# 1. Kiểm tra "tình hình chiến sự" trước khi cân bằng
print(f"Số lượng mẫu trước khi SMOTE: {Counter(y_train_fixed)}")

# 2. Khởi tạo SMOTE
# sampling_strategy='auto' sẽ đưa tất cả các app ít về bằng số lượng app nhiều nhất
smote = SMOTE(sampling_strategy='auto', k_neighbors=3, random_state=42)

# 3. Tiến hành "nhân bản thông minh"
# Lưu ý: Chỉ thực hiện trên tập TRAIN (X_train_scaled, y_train_fixed)
print("Đang chạy SMOTE... ")
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train_fixed)

# 4. Kiểm tra lại kết quả
print(f"Số lượng mẫu SAU khi SMOTE: {Counter(y_train_resampled)}")
print(f"Tổng số dòng dữ liệu mới: {len(X_train_resampled)}")

NameError: name 'y_train_fixed' is not defined

In [4]:
#ô code 4
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.layers import BatchNormalization

In [5]:
# ô code 5 - ĐÃ SỬA ĐỂ DÙNG DỮ LIỆU SMOTE
import numpy as np

# dùng X_train_resampled (dữ liệu sau SMOTE)
X_train_cnn = np.expand_dims(X_train_resampled, axis=2)

# X_test thì vẫn giữ nguyên X_test_fixed (vì tập test không được SMOTE)
X_test_cnn = np.expand_dims(X_test_fixed, axis=2)

print(f"✅ Cấu trúc dữ liệu 3D đã sẵn sàng (Đã bao gồm SMOTE): {X_train_cnn.shape}")

Cấu trúc dữ liệu 3D cho CNN: (80000, 80, 1)


In [6]:
# ô code 6
num_features = X_train_scaled.shape[1] # Tự động lấy số lượng cột (đặc trưng)
num_classes = len(le_final.classes_)   # Tự động lấy số lượng ứng dụng

model = Sequential([
    # Lớp lọc đặc trưng 1(Convolutional Layer)
    Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(), #chuẩn hóa theo lô (giúp dữ liệu ổn định hơn)
    MaxPooling1D(2),
    #Lớp lọc đặc trưng 2
    Conv1D(128, 3, activation='relu', input_shape=(num_features, 1)),
    BatchNormalization(),
    MaxPooling1D(2),
    Dropout(0.2), #giảm 20% neuron để tránh học vẹt
    # Lớp làm phẳng dữ liệu để đưa vào mạng nơ-ron
    Flatten(),
    Dense(256, activation='relu'), #256 neuron
    Dropout(0.4), # drop thêm neuron để dữ liệu chạy ổn định
    # Lớp đầu ra (Số lượng app thực tế)
    Dense(num_classes, activation='softmax') # Khớp với số lượng app thực tế
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 78, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 78, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 39, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 37, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 37, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 50)             │        12,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 628,658 (2.40 MB)

 Trainable params: 628,274 (2.40 MB)

 Non-trainable params: 384 (1.50 KB)

In [7]:
# ô code 7
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Tự động dừng nếu 5 vòng không tăng accuracy (tiết kiệm thời gian cho Mạnh)
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

# 2. Tự động giảm tốc độ học nếu bị "kẹt" (giúp mô hình học sâu hơn)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
# Sử dụng y_train_fixed và y_test_fixed để khớp với mã hóa của XGBoost ở trên
history = model.fit(
    X_train_cnn,
    y_train_fixed,
    epochs=50,
    batch_size=128,
    validation_data=(X_test_cnn, y_test_fixed),
    callbacks=[early_stop, reduce_lr] # hàm gọi lại khi hết 1 epoch git kiểm tra mô hình có tốt lên ko ? có nên dùng ko
)

Epoch 1/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.5196 - loss: 1.5982 - val_accuracy: 0.5684 - val_loss: 1.3649 - learning_rate: 0.0010
Epoch 2/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5832 - loss: 1.3456 - val_accuracy: 0.6201 - val_loss: 1.2140 - learning_rate: 0.0010
Epoch 3/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6068 - loss: 1.2639 - val_accuracy: 0.6371 - val_loss: 1.1471 - learning_rate: 0.0010
Epoch 4/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6187 - loss: 1.2161 - val_accuracy: 0.6418 - val_loss: 1.1485 - learning_rate: 0.0010
Epoch 5/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6279 - loss: 1.1815 - val_accuracy: 0.6500 - val_loss: 1.1195 - learning_rate: 0.0010
Epoch 6/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6327 - loss: 1.1585 - val_accuracy: 0.6592 - val_loss: 1.0795 - learning_rate: 0.0010
Epoch 7/50
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6409 - loss: 1.1377 -

In [8]:
from sklearn.metrics import classification_report
import numpy as np

# 1. Dự đoán lớp
y_pred = model.predict(X_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)

# 2. Lấy các nhãn thực tế xuất hiện (tránh lỗi 40 vs 54)
present_labels = np.unique(np.concatenate([y_test_fixed, y_pred_classes]))

# 3. ÉP KIỂU SANG CHUỖI (Sửa lỗi TypeError ở ảnh f765ee)
present_names = [str(le_final.classes_[i]) for i in present_labels]

# 4. In báo cáo chi tiết
print(classification_report(y_test_fixed, y_pred_classes,
                            labels=present_labels,
                            target_names=present_names))

625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
              precision    recall  f1-score   support

           0       0.65      0.13      0.22       280
           1       0.75      0.08      0.14        39
           2       0.33      0.11      0.17         9
           3       0.00      0.00      0.00         9
           5       0.86      0.27      0.41        70
           6       1.00      1.00      1.00         7
           8       0.55      0.67      0.60         9
           9       0.92      0.68      0.78       146
          10       0.00      0.00      0.00         1
          11       0.00      0.00      0.00         5
          12       0.86      0.64      0.73       130
          13       0.00      0.00      0.00         1
          14       0.50      0.02      0.04       165
          15       0.66      0.78      0.71      4884
          16       0.00      0.00      0.00         3
          17       0.79      0.93      0.85      4845
          18       0.66      0.67      0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
